In [1]:
from pathlib import Path
import json, math
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C

METHODS = ["additive","plldelta","mutant_ctx"]
ROUND0_DIR = Path(Path.cwd().parent / "round0")   # adjust if needed
OUT_DIR = Path("cache/models"); OUT_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 5
PCA_DIMS = 64
RANDOM_STATE = 7

def spearmanr_fast(a, b):
    """Spearman without SciPy: Pearson of ranks."""
    ra = pd.Series(a).rank(method="average").to_numpy()
    rb = pd.Series(b).rank(method="average").to_numpy()
    ra = (ra - ra.mean()) / (ra.std() + 1e-12)
    rb = (rb - rb.mean()) / (rb.std() + 1e-12)
    return float(np.mean(ra * rb))

def coverage_fraction(y, mu, sigma, z=1.0):
    """Fraction of targets within z*sigma of mean."""
    return float(np.mean(np.abs(y - mu) <= z * sigma))

def nll_gaussian(y, mu, sigma):
    s2 = np.maximum(sigma**2, 1e-12)
    return float(np.mean(0.5*np.log(2*math.pi*s2) + 0.5*((y-mu)**2)/s2))

def pct_at_bounds(length_scales, lo=1e-2, hi=1e2, tol=1.02):
    """Percent of ARD Ls near lower/upper bounds (within a factor tol)."""
    ls = np.asarray(length_scales, dtype=float)
    hi_hit = np.mean(ls >= hi / tol) if ls.size else 0.0
    lo_hit = np.mean(ls <= lo * tol) if ls.size else 0.0
    return float(lo_hit), float(hi_hit)

def cv_eval_gp_npz(npz_path):
    data = np.load(npz_path, allow_pickle=True)
    X0 = data["X"].astype(np.float32)
    y  = data["y_norm"].astype(np.float32)
    alpha_all = data["alpha"].astype(np.float32) if "alpha" in data.files else None

    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    metrics = {
        "rmse": [], "mae": [], "pearson_r": [], "spearman_rho": [],
        "cov_68": [], "cov_95": [], "sharpness_sigma": [], "nll": [],
        "ls_low_pct": [], "ls_high_pct": []
    }

    for fold, (tr, te) in enumerate(kf.split(X0), 1):
        Xtr, Xte = X0[tr], X0[te]
        ytr, yte = y[tr], y[te]
        alpha = alpha_all[tr] if alpha_all is not None else 1e-6

        # same featurization as in training recipe
        sc1 = StandardScaler().fit(Xtr);     Xtr1 = sc1.transform(Xtr);   Xte1 = sc1.transform(Xte)
        pca = PCA(n_components=PCA_DIMS, random_state=RANDOM_STATE).fit(Xtr1)
        Xtr2 = pca.transform(Xtr1);          Xte2 = pca.transform(Xte1)
        sc2 = StandardScaler().fit(Xtr2);    Xtrf = sc2.transform(Xtr2);  Xtef = sc2.transform(Xte2)

        d = Xtrf.shape[1]
        kernel = C(1.0, (1e-2, 1e2)) * Matern(length_scale=np.ones(d),
                                              length_scale_bounds=(1e-2, 1e2), nu=2.5) \
                 + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-1))

        gp = GaussianProcessRegressor(kernel=kernel, alpha=alpha, normalize_y=False,
                                      n_restarts_optimizer=5, random_state=RANDOM_STATE)
        gp.fit(Xtrf, ytr)

        mu, sigma = gp.predict(Xtef, return_std=True)
        resid = yte - mu
        rmse = float(np.sqrt(np.mean(resid**2)))
        mae  = float(np.mean(np.abs(resid)))
        pear = float(np.corrcoef(mu, yte)[0,1]) if len(yte)>1 else np.nan
        spear= spearmanr_fast(mu, yte) if len(yte)>1 else np.nan
        cov68= coverage_fraction(yte, mu, sigma, z=1.0)
        cov95= coverage_fraction(yte, mu, sigma, z=1.96)
        sharp= float(np.mean(sigma))
        nll  = nll_gaussian(yte, mu, sigma)

        # ARD diagnostics
        k = gp.kernel_
        try:
            ls = k.k1.k2.length_scale  # Constant * Matern + White
        except Exception:
            ls = np.array([])
        lo_pct, hi_pct = pct_at_bounds(ls)

        for kname, val in [
            ("rmse",rmse), ("mae",mae), ("pearson_r",pear), ("spearman_rho",spear),
            ("cov_68",cov68), ("cov_95",cov95), ("sharpness_sigma",sharp), ("nll",nll),
            ("ls_low_pct",lo_pct), ("ls_high_pct",hi_pct),
        ]:
            metrics[kname].append(val)

    # summarize
    summary = {k: {"mean": float(np.nanmean(v)), "std": float(np.nanstd(v))} for k,v in metrics.items()}
    return summary

if __name__ == "__main__":
    all_results = {}
    for tag in METHODS:
        npz = ROUND0_DIR / f"round0_{tag}_train_matrix.npz"
        if not npz.exists():
            print(f"[{tag}] missing {npz.name}, skipping.")
            continue
        res = cv_eval_gp_npz(npz)
        all_results[tag] = res
        # write per-method json
        with open(OUT_DIR / f"round0_{tag}_cv.json","w") as f:
            json.dump(res, f, indent=2)
        print(f"\n[{tag}] 5-fold CV:")
        print(f"  RMSE (y_norm): {res['rmse']['mean']:.3f} ± {res['rmse']['std']:.3f}")
        print(f"  MAE  (y_norm): {res['mae']['mean']:.3f}")
        print(f"  Pearson r     : {res['pearson_r']['mean']:.3f}")
        print(f"  Spearman rho  : {res['spearman_rho']['mean']:.3f}")
        print(f"  Coverage 68%  : {res['cov_68']['mean']:.3f} (target ~0.68)")
        print(f"  Coverage 95%  : {res['cov_95']['mean']:.3f} (target ~0.95)")
        print(f"  Sharpness σ   : {res['sharpness_sigma']['mean']:.3f} (smaller=sharper)")
        print(f"  NLL           : {res['nll']['mean']:.3f} (lower=better)")
        print(f"  ARD @low/hi % : {100*res['ls_low_pct']['mean']:.1f}% / {100*res['ls_high_pct']['mean']:.1f}%")


C:\Users\thana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\thana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\thana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: Th


[additive] 5-fold CV:
  RMSE (y_norm): 4.029 ± 0.662
  MAE  (y_norm): 2.058
  Pearson r     : 0.453
  Spearman rho  : 0.349
  Coverage 68%  : 0.876 (target ~0.68)
  Coverage 95%  : 0.876 (target ~0.95)
  Sharpness σ   : 4.167 (smaller=sharper)
  NLL           : 2.249 (lower=better)
  ARD @low/hi % : 0.0% / 0.0%


C:\Users\thana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\gaussian_process\_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 11 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
C:\Users\thana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\thana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\gaussian_process\kernels.py:4


[plldelta] 5-fold CV:
  RMSE (y_norm): 3.988 ± 1.403
  MAE  (y_norm): 2.034
  Pearson r     : 0.135
  Spearman rho  : 0.308
  Coverage 68%  : 0.933 (target ~0.68)
  Coverage 95%  : 0.944 (target ~0.95)
  Sharpness σ   : 4.040 (smaller=sharper)
  NLL           : 2.313 (lower=better)
  ARD @low/hi % : 0.3% / 0.0%


C:\Users\thana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\thana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\thana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: Th


[mutant_ctx] 5-fold CV:
  RMSE (y_norm): 5.628 ± 1.207
  MAE  (y_norm): 2.949
  Pearson r     : 0.118
  Spearman rho  : 0.236
  Coverage 68%  : 0.853 (target ~0.68)
  Coverage 95%  : 0.864 (target ~0.95)
  Sharpness σ   : 5.470 (smaller=sharper)
  NLL           : 2.592 (lower=better)
  ARD @low/hi % : 0.0% / 0.3%


C:\Users\thana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
